# Fashion-MNIST — ARPG Pipeline (Baseline Hazır)

**Bu notebook baseline train ETMEZ.** PixelCNN++ 20 epoch sonuçları Google Drive'da:
`My Drive/comp547_outputs/pixelcnnpp_fashion_e20/`

## Akış
1. Setup + Drive'dan baseline restore + doğrula
2. ARPG train (20 epoch, checkpoint + resume)
3. K-sweep (33 koşul)
4. Sunum grafikleri
5. FID (final report)
6. Drive final yedek

## Kopma sonrası (ARPG)
1. Cell 1–4 (setup + baseline restore)
2. **Cell 9** (ARPG Drive'dan otomatik geri yukle)
3. **Cell 10** (train) → `resuming_from=last.pt` ile kaldigi epoch'tan devam

Her epoch sonunda Drive'a otomatik yedek: `comp547_outputs/arpg_fashion_latest/`

## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/comp547_outputs'
BASELINE_DRIVE = f'{DRIVE_BASE}/pixelcnnpp_fashion_e20'
ARPG_DRIVE = f'{DRIVE_BASE}/arpg_fashion'
!mkdir -p "{DRIVE_BASE}"

In [ ]:
%cd /content
!rm -rf COMP547PROJECT
!git clone https://github.com/oaydogdu/COMP547PROJECT.git
%cd COMP547PROJECT
!git pull origin main
!pip install -q -r requirements.txt

In [ ]:
import sys, json, torch
from pathlib import Path
sys.path.insert(0, 'src')

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()

from ARPG.arpg_runner import train_arpg, run_arpg_sweep
from common.checkpointing import backup_results_tree
print('Imports OK')

## 1 — Baseline restore (TRAIN YOK)

Drive'daki tamamlanmış PixelCNN++ sonuçlarını Colab'a kopyalar.

In [ ]:
import shutil
from pathlib import Path

%cd /content/COMP547PROJECT

BASELINE_LOCAL = Path('results/pixelcnnpp_fashion_e20')
BASELINE_DRIVE = Path('/content/drive/MyDrive/comp547_outputs/pixelcnnpp_fashion_e20')

if not BASELINE_DRIVE.exists():
    raise FileNotFoundError(
        f'Baseline Drive klasoru yok: {BASELINE_DRIVE}\n'
        'Drive yolunu kontrol et.'
    )

if BASELINE_LOCAL.exists():
    shutil.rmtree(BASELINE_LOCAL)
shutil.copytree(BASELINE_DRIVE, BASELINE_LOCAL)
print('Restored ->', BASELINE_LOCAL)

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

root = Path('results/pixelcnnpp_fashion_e20')
required = {
    'checkpoint': list((root / 'checkpoints').glob('*.pt')),
    'fashion_eval.json': root / 'eval' / 'fashion_eval.json',
    'fashion_grid.png': root / 'eval' / 'fashion_grid.png',
    'metrics.json': sorted((root / 'metrics').glob('*.json')),
}

ok = True
for name, val in required.items():
    exists = bool(val) if isinstance(val, list) else val.exists()
    print(f"{'OK' if exists else 'MISSING':7s}  {name}")
    ok = ok and exists

assert ok, 'Baseline dosyalari eksik — train gerekmez ama Drive yedegini kontrol et.'
print('\nBaseline TRAIN GEREKMIYOR — mevcut sonuclar kullanilacak.')

ev = json.loads((root / 'eval' / 'fashion_eval.json').read_text())
print('\nBaseline ozet:')
print(f"  latency_ms_per_image = {ev['latency_ms_per_image']:.2f}")
print(f"  throughput_img_per_s = {ev['throughput_img_per_s']:.3f}")

display(Image(str(root / 'eval' / 'fashion_grid.png'), width=400))
if (root / 'eval' / 'real_vs_generated.png').exists():
    display(Image(str(root / 'eval' / 'real_vs_generated.png'), width=500))

## 2 — ARPG train (20 epoch, otomatik resume + Drive yedek)

**Kopma sonrasi:** Cell 9 → Cell 10 (train tekrar). Epoch 1'den baslamaz.

In [ ]:
# Her train oncesi calistir: Drive'da yedek varsa geri yukler, yoksa 'fresh' der
%cd /content/COMP547PROJECT
import sys
sys.path.insert(0, 'src')
from common.checkpointing import restore_arpg_for_resume

DRIVE_BASE = '/content/drive/MyDrive/comp547_outputs'
status = restore_arpg_for_resume('results/arpg_fashion', DRIVE_BASE)
print('ARPG resume status:', status)

In [ ]:
%cd /content/COMP547PROJECT
!git pull origin main
!PYTHONPATH=src python scripts/train_arpg.py \
  --dataset fashion_mnist \
  --data-dir data \
  --save-dir results/arpg_fashion \
  --epochs 20 \
  --batch-size 16 \
  --d-model 192 \
  --n-heads 6 \
  --n-layers 6 \
  --num-workers 0 \
  --save-every-epochs 5 \
  --seed 1 \
  --drive-backup-dir /content/drive/MyDrive/comp547_outputs/arpg_fashion_latest

In [ ]:
from pathlib import Path
ckpt_dir = Path('results/arpg_fashion/checkpoints')
for name in ['last.pt', 'best.pt']:
    print(name, (ckpt_dir / name).exists())
print('epoch ckpts:', sorted(ckpt_dir.glob('epoch_*.pt')))

## 3 — ARPG K-sweep (K x schedule)

In [ ]:
from pathlib import Path

# Kopma sonrasi: once ARPG Drive yedegini restore et (asagidaki cell) sonra buraya gel
ckpt_dir = Path('results/arpg_fashion/checkpoints')
ckpt = ckpt_dir / 'best.pt'
if not ckpt.exists():
    ckpt = sorted(ckpt_dir.glob('*.pt'))[-1]
print('ARPG checkpoint:', ckpt)

!PYTHONPATH=src python scripts/eval_arpg.py \
  --checkpoint "{ckpt}" \
  --out-dir results/arpg_fashion/eval \
  --ks 1,2,4,7,14,28,56,112,196,392,784 \
  --schedules random,raster,row \
  --n-samples 25 \
  --seed 42 \
  --top-p 0.9 \
  --temperature 1.0

## 4 — Sunum grafikleri

In [ ]:
import glob
metrics = sorted(glob.glob('results/pixelcnnpp_fashion_e20/metrics/*.json'))
metrics_path = metrics[-1] if metrics else ''

!PYTHONPATH=src python scripts/build_fashion_report.py \
  --baseline-eval-json results/pixelcnnpp_fashion_e20/eval/fashion_eval.json \
  --arpg-sweep-json results/arpg_fashion/eval/sweep.json \
  --baseline-metrics-json "{metrics_path}" \
  --out-dir results/fashion_presentation

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import json

display(Image('results/fashion_presentation/tradeoff_speed.png', width=700))
for sched in ['random', 'raster', 'row']:
    p = f'results/fashion_presentation/quality_strip_{sched}.png'
    if Path(p).exists():
        display(Image(p, width=700))

print(json.dumps(json.load(open('results/fashion_presentation/presentation_summary.json')), indent=2))

## 5 — FID (final report)

Train gerekmez. Checkpoint'ten 2048 goruntu uret + FID hesapla.
Her kosul ~15-30 dk (baseline + 3 ARPG K degeri onerilir).

In [ ]:
%cd /content/COMP547PROJECT

BASELINE_CKPT = 'results/pixelcnnpp_fashion_e20/checkpoints/pixelcnnpp_fashion_mnist_lr0.00020_res5_f160.pt'

!PYTHONPATH=src python scripts/compute_fid_fashion.py \
  --model baseline \
  --checkpoint "{BASELINE_CKPT}" \
  --out-dir results/fid/baseline \
  --n-samples 2048 \
  --compute-fid \
  --out-json results/fid/baseline_fid.json

In [ ]:
import pathlib
arpg_ckpt = 'results/arpg_fashion/checkpoints/best.pt'
if not pathlib.Path(arpg_ckpt).exists():
    arpg_ckpt = sorted(pathlib.Path('results/arpg_fashion/checkpoints').glob('*.pt'))[-1]

for k in [1, 28, 784]:
    out = f'results/fid/arpg_random_K{k}'
    json_out = f'results/fid/arpg_random_K{k}_fid.json'
    !PYTHONPATH=src python scripts/compute_fid_fashion.py \
      --model arpg \
      --checkpoint "{arpg_ckpt}" \
      --out-dir {out} \
      --k {k} \
      --schedule random \
      --n-samples 2048 \
      --compute-fid \
      --out-json {json_out}

## 6 — Final Drive yedek

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

final = Path('/content/drive/MyDrive/comp547_outputs') / f"final_{datetime.now().strftime('%Y%m%d_%H%M')}"
final.mkdir(parents=True, exist_ok=True)

for name in ['pixelcnnpp_fashion_e20', 'arpg_fashion', 'fashion_presentation', 'fid']:
    src = Path('results') / name
    if src.exists():
        dst = final / name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('saved', dst)

print('FINAL BACKUP:', final)

In [ ]:
# Kopma sonrasi ARPG restore (baseline icin Cell 4 yeterli)
import shutil
from pathlib import Path

src = Path('/content/drive/MyDrive/comp547_outputs')
candidates = sorted(src.glob('arpg_*'), reverse=True)
if not candidates:
    print('ARPG Drive yedegi yok — train cell ile basla')
else:
    latest = candidates[0] / 'arpg_fashion' if (candidates[0] / 'arpg_fashion').exists() else candidates[0]
    dst = Path('results/arpg_fashion')
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(latest, dst)
    print('ARPG restored from', latest)